# 13 · Phase 1 · E1.3: H4 conv ablation  (~0.9 GPU-h, 12 runs)

A2 with {Engram conv off}, {GDN short conv off} and {both off}, plus A1 with the GDN short conv off, at the primary cell
(N\*, F=512, P=64, a, λ=1), 3 seeds. Full A2 and A1 come from E1.2 / E1.1.

In [ ]:
# ===================== SETTINGS: edit this cell, then "Run All" (or "Save Version -> Save & Run All") =====================
REPO_URL     = "https://github.com/Shiveshrane/Engram.git"   # None -> copy the code from CODE_DATASET instead
REPO_REF     = "main"           # pin the PREREG commit hash for Phase 1 / 2 runs
CODE_DATASET = "/kaggle/input/engramdelta-src"   # Kaggle Dataset with the repo (used when REPO_URL is None)
HF_REPO      = None             # e.g. "your-hf-user/engramdelta-runs" (private dataset repo; needs Kaggle Secret HF_TOKEN)
GDN_BACKEND  = "torch"          # "torch" (pure PyTorch, always works) or "fla" (only if NB00 says PASS)
FLA_SPEC     = "fla-core==0.5.2"                 # FLA's kernel package (fla.ops); installed with --no-deps
PREV_OUTPUTS = []               # outputs of earlier sessions attached as inputs, e.g. ["/kaggle/input/engramdelta-11-phase1-h1-grid"]
NCCL_ENV     = {}               # set to {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"} if NB00's first NCCL test hung

In [ ]:
# ===================== Session setup (no edits needed) =====================
import os, sys, subprocess, shutil, pathlib, json, glob, time
NEEDS_GPU, INSTALL_FLA = True, False
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path.cwd()
# Locally (notebook opened from the repo's notebooks/ folder) the repo is the parent directory.
REPO = WORK / "engramdelta" if ON_KAGGLE else (WORK.parent if WORK.name == "notebooks" else WORK)

def run(cmd, check=True, env=None, timeout=None, show=None):
    """Run a shell command in the repo and stream its output into this notebook."""
    print("$", show or cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})}, cwd=str(REPO) if REPO.exists() else str(WORK))
    t0 = time.time()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if timeout and time.time() - t0 > timeout:
                raise subprocess.TimeoutExpired(cmd, timeout)
        rc = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        p.kill(); print(f"!! timed out after {timeout}s"); rc = -9
    if check and rc != 0:
        raise RuntimeError(f"exit code {rc}: {show or cmd}")
    return rc

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

# 1. code
if not REPO.exists():
    if REPO_URL:
        url, tok = REPO_URL, secret("GITHUB_TOKEN")
        if tok and url.startswith("https://"):
            url = url.replace("https://", f"https://{tok}@")
        run(f"git clone -q {url} {REPO}", show=f"git clone {REPO_URL}")
        run(f"git -C {REPO} checkout -q {REPO_REF}")
    else:
        shutil.copytree(CODE_DATASET, REPO)
os.chdir(REPO); sys.path.insert(0, str(REPO))
try:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    commit = open("GIT_COMMIT").read().strip() if os.path.exists("GIT_COMMIT") else "unknown"
os.environ["GIT_COMMIT"] = commit
print("code commit:", commit)

# 2. packages (never let pip replace Kaggle's torch)
run("pip install -q --no-deps einops sympy")
if INSTALL_FLA or GDN_BACKEND == "fla":
    run(f"pip install -q --no-deps {FLA_SPEC}", check=False)

# 3. environment for every launched run
os.environ["ENGRAMDELTA_GDN_BACKEND"] = GDN_BACKEND
os.environ["ENGRAMDELTA_CKPT_DIR"] = str(WORK / "ckpt")
os.environ.update(NCCL_ENV)
if HF_REPO and secret("HF_TOKEN"):
    os.environ["HF_REPO"], os.environ["HF_TOKEN"] = HF_REPO, secret("HF_TOKEN")
    print("HF Hub sync: ON ->", HF_REPO)
else:
    print("HF Hub sync: OFF (results persist via this notebook's output only)")

# 4. restore results / checkpoints from earlier sessions (never overwrites existing files)
def restore(src, dst):
    if not os.path.isdir(src):
        return 0
    n = 0
    for root, _, files in os.walk(src):
        for f in files:
            s = os.path.join(root, f); d = os.path.join(dst, os.path.relpath(s, src))
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d); n += 1
    return n
for prev in PREV_OUTPUTS:
    n1 = restore(os.path.join(prev, "engramdelta", "results"), str(REPO / "results"))
    n2 = restore(os.path.join(prev, "ckpt"), str(WORK / "ckpt"))
    print(f"restored from {prev}: {n1} result files, {n2} checkpoint files")
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync pull results", check=False)

# 5. hardware
if NEEDS_GPU:
    run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
NPROC = max(1, torch.cuda.device_count())
if NEEDS_GPU and NPROC < 2:
    print("!! WARNING: fewer than 2 GPUs. Set Accelerator = 'GPU T4 x2' in the notebook settings.")

In [ ]:
GROUPS = ['E1.3']     # run in this order (later groups may need earlier results, e.g. LRs from E1.0a)
SESSION_HOURS = 2.0   # leave margin below Kaggle's 12 h session limit
EXTRA = ""
t_start = time.time()
for g in GROUPS:
    left = SESSION_HOURS - (time.time() - t_start) / 3600
    if left < 0.15:
        print("session budget used up; rerun this notebook to continue"); break
    run(f"python -m engramdelta.scripts.make_manifest --group {g} --out manifests {EXTRA}")
    run(f"python -m engramdelta.scripts.budget manifests/{g}.yaml --nproc {NPROC}", check=False)
    rc = run(f"python -m engramdelta.scripts.sweep manifests/{g}.yaml --nproc {NPROC} --session-hours {left:.2f}", check=False)
    if rc != 0:
        print(f"!! sweep for {g} reported failures or stopped early (exit {rc}). Check results/_failures before continuing.")

In [ ]:
import yaml
from engramdelta import analysis
pr = yaml.safe_load(open("configs/prereg.yaml"))
print(analysis.h4_table(analysis.load("results"), pr).to_string(index=False))

In [ ]:
# ===================== End of session: persist =====================
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync push results", check=False)
n = sum(len(f) for _, _, f in os.walk(REPO / "results"))
print(f"{n} files under results/. They are part of this notebook's output: Save Version keeps them.")
print("Next session: attach this notebook's output as an input and add its path to PREV_OUTPUTS (or use HF_REPO).")